In [ ]:
#!pip install openai powerlaw scipy matplotlib seaborn pandas

import openai
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import powerlaw
from scipy import stats
from tqdm import tqdm

# Insert your OpenAI API key below
# Example: api_key="your_api_key_here"
client = openai.OpenAI(api_key="")

In [ ]:
PROBLEMS = {
    "APHCE": {  # Annual Patient Health-Care Expenditures
        "natural": (
            "Generate a synthetic dataset of 1000 patients with their annual health-care expenditures in USD."
        ),
        "mixed": (
            "Generate a synthetic dataset of 1000 patients with annual health-care expenditures in USD reflecting real-world variability, where most expenditures are moderate but a few are very high. "
        ),
        "controlled": (
            "Generate a synthetic dataset of 1000 fictitious patients along with their annual health-care expenditures in USD that follow a power-law distribution. Ensure that most patients have moderate expenditures while a few incur very high costs. "
        )
    },
   

"PRESC": {  # Prescription Frequency by Drug Class

    "natural": (

        "Generate a synthetic dataset of 1000 Anatomical Therapeutic Chemical drug classes with their annual prescription counts. "

    ),

    "mixed": (

        "Generate a synthetic dataset of 1000 Anatomical Therapeutic Chemical drug classes with annual prescription counts reflecting real-world variability, where most classes have low volumes but a few have very high volumes. "

    ),

    "controlled": (

        "Generate a synthetic dataset of 1000 Anatomical Therapeutic Chemical drug classes with annual prescription counts that follow a power-law distribution.Ensure that most classes have modest counts while a few have very high counts."

    )

},


    "DDI": {  # Drug–Drug Interaction (DDI) Network Degree
        "natural": (
            "Generate a synthetic dataset of 1000 drugs by name with their number of known drug–drug interaction partners. "
        ),
        "mixed": (
            "Generate a synthetic dataset of 1000 drugs by name with their number of drug–drug interaction partners reflecting real-world networks, where most drugs have few interactions but a few serve as highly connected hubs. "
        ),
        "controlled": (
           "Generate a synthetic dataset of 1000 drugs by name, with counts of drug–drug interaction partners that follow a power-law distribution. Ensure that most drugs have few partners while a few have very many. "
        )
    },
    "GEA": {  # Gene-Expression Abundance
        "natural": (
            "Generate a synthetic dataset of 1000 genes with their transcript counts from an RNA-seq experiment. "
        ),
        "mixed": (
            "Generate a synthetic dataset of 1000 fictitious genes with transcript counts reflecting real-world RNA-seq variability, where most genes have low expression and a few are highly expressed. "
        ),
        "controlled": (
            "Generate a synthetic dataset of 1000 fictitious genes whose transcript counts follow power law distribution. Ensure that most genes have low counts while a few have very high counts. "
        )
    },
    "PPI": {  # Protein–Protein Interaction Network Degree
        "natural": (
            "Generate a synthetic dataset of 1000 proteins by UniProt ID with their number of interaction partners in a PPI network. "
        ),
        "mixed": (
            "Generate a synthetic dataset of 1000 proteins by UniProt ID with PPI degrees reflecting real-world networks, where most proteins have few partners but a few serve as hubs. "
        ),
        "controlled": (
            "Generate a synthetic dataset of 1000 proteins by UniProt ID whose PPI degrees follow a power-law distribution. Ensure that most proteins have few interactions while a few have many. "
        )
    },
    "METC": {  # Metabolite Connectivity in Metabolic Networks
        "natural": (
            "Generate a synthetic dataset of 1000 metabolites by name with their number of associated reactions."
        ),
        "mixed": (
            "Generate a synthetic dataset of 1000 metabolites by name with reaction counts reflecting metabolic network variability, where most metabolites appear in few reactions but a few are highly connected. "
        ),
        "controlled": (
            "Generate a synthetic dataset of 1000 metabolites by name whose reaction counts follow a power-law distribution. Ensure that most metabolites have few reactions while a few have many."
        )
    }
}

In [ ]:
LLMS = ["gpt-4o", "o4-mini", "o3", "gpt-4.1"]
TEMPS = [0, 0.3, 0.7, 1.0]
REPEATS = 5
TOP_P = [0.9, 0.95, 1.0]

In [ ]:
import io, os
import pandas as pd

def get_csv_from_gpt(problem, typ, model, temperature, top_p, num, rep):
    system_msg = {
        "role": "system",
        "content": "Please generate a CSV table with exactly {} rows plus one header with two columns: id and numbers, no skipped rows, using comma separators, and no markdown formatting. You must complete the full, regardless of how long it takes. Do not stop, truncate, or summarize — generate the entire output to the end.".format(num)
    }

    user_msg = {"role": "user", "content": PROBLEMS[problem][typ.lower()]}

    try:
        if model == "gpt-4o" or model == "gpt-4.1" :
          response = client.chat.completions.create(model=model, messages=[system_msg, user_msg], temperature=temperature, top_p=top_p)
        else:
          response = client.chat.completions.create(model=model, messages=[system_msg, user_msg])

        content = response.choices[0].message.content
        df_csv = pd.read_csv(io.StringIO(content))

        print(df_csv)

        folder_name_for_model = {"gpt-4o": "G4o", "gpt-4.1": "G41", "o3":"o3", "o4-mini":"o4M"}
        folder_name_for_problem = {"APHCE": "A", "PFDC": "P", "DDIND": "D", "GEA": "G", "PPIND": "PP", "MCMN": "M"}
        acronym_for_type = {"natural": "N", "mixed": "M", "controlled": "C"}

        full_path = os.path.join(model, problem)
        os.makedirs(full_path, exist_ok=True)

        if model == "gpt-4o" or model == "gpt-4.1" :
          file_name = "{}_{}_{}_{}_{}_{}_{}.csv".format(
              folder_name_for_model.get(model, model),
              folder_name_for_problem.get(problem, problem),
              num,
              acronym_for_type.get(typ.lower(), typ),
              int(temperature * 100),
              int(top_p * 100),
              rep
          )
        else:
          file_name = "{}_{}_{}_{}_{}.csv".format(
              folder_name_for_model.get(model, model),
              folder_name_for_problem.get(problem, problem),
              num,
              acronym_for_type.get(typ.lower(), typ),
              rep
          )

        file_path = os.path.join(full_path, file_name)

        df_csv.to_csv(file_path, index=False)


    except Exception as e:
        print(f" API error: {e}")
        return None

In [ ]:
for problem, styles in PROBLEMS.items():
    for typ, prompt in styles.items():
        for llm in LLMS:
            if llm=="gpt-4o" or llm=="gpt-4.1":
                # Vary both temperature and top_p
                for temperature in TEMPS:
                    for top_p in TOP_P:
                        for rep in range(1, REPEATS + 1):
                            print(f" {problem} | {typ} | {llm} | temp={temperature} | top_p={top_p} | rep={rep}")
                            get_csv_from_gpt(problem, typ, llm, temperature, top_p, 1000, rep)
            else:
                # Use default temperature/top_p (e.g. temperature=0, top_p=1.0)
                temperature = 0
                top_p = 1.0
                for rep in range(1, REPEATS + 1):
                    print(f" {problem} | {typ} | {llm} | rep={rep}")
                    get_csv_from_gpt(problem, typ, llm, temperature, top_p, 1000, rep)